# 8.3 实验一：自主定点悬停（推理）

> **AirSim 配置**：使用 `settings.json`（单架无人机），复制到 `~/Documents/AirSim/settings.json` 后启动 AirSim。

## 任务目标

让无人机在指定位置 (0, 0, -5) 保持稳定悬停，抵抗环境扰动。

本节使用预训练好的世界模型权重进行推理演示。如果没有预训练权重，我们提供一个基于简化世界模型的 baseline 策略作为对比。

In [ ]:
import sys
sys.path.append('../external-libraries')
sys.path.append('.')

import numpy as np
import matplotlib.pyplot as plt
import airsim
import time
from PIL import Image

# 连接 AirSim
client = airsim.MultirotorClient()
client.confirmConnection()
print(f"连接成功！无人机: {client.listVehicles()}")

## 8.3.1 奖励函数设计

奖励函数是强化学习的核心——它定义了"什么是好的行为"：

| 条件 | 奖励 | 含义 |
|------|------|------|
| 保持在目标位置附近 | +1.0/step | 鼓励稳定悬停 |
| 姿态倾斜过大 | -0.1/step | 惩罚不稳定 |
| 坠机或飞出边界 | -10.0 | 严厉惩罚危险行为 |

![悬停任务场景](figures/hover_scene.png)

*图 8-7：AirSim 中的悬停任务场景——无人机需要在指定位置保持稳定*

In [ ]:
# 定义悬停任务参数
TARGET = np.array([0.0, 0.0, -5.0])  # 目标位置 (NED坐标，z=-5表示5米高)
DRONE = "Drone1"

def get_drone_state(client, drone_id):
    """获取无人机完整状态。"""
    state = client.getMultirotorState(vehicle_name=drone_id)
    pos = state.kinematics_estimated.position
    vel = state.kinematics_estimated.linear_velocity
    ori = state.kinematics_estimated.orientation
    return {
        'pos': np.array([pos.x_val, pos.y_val, pos.z_val]),
        'vel': np.array([vel.x_val, vel.y_val, vel.z_val]),
        'ori': np.array([ori.w_val, ori.x_val, ori.y_val, ori.z_val]),
    }

def compute_hover_reward(state, target):
    """计算悬停奖励。"""
    dist = np.linalg.norm(state['pos'] - target)
    tilt = abs(state['ori'][1]) + abs(state['ori'][2])  # roll + pitch
    reward = max(0, 1.0 - dist / 5.0) - 0.1 * tilt
    return float(reward), dist

print(f"目标位置: {TARGET}")
print(f"奖励函数: reward = max(0, 1 - dist/5) - 0.1 * tilt")

## 8.3.2 Baseline：随机动作 vs 简单PD控制

在加载世界模型之前，我们先看两个 baseline 的表现，作为对比参照。

In [ ]:
def run_episode(client, policy_fn, target, n_steps=200, label=""):
    """运行一个 episode，记录轨迹和奖励。"""
    client.reset()
    client.enableApiControl(True, vehicle_name=DRONE)
    client.armDisarm(True, vehicle_name=DRONE)
    client.takeoffAsync(vehicle_name=DRONE).join()
    client.moveToPositionAsync(target[0], target[1], target[2], 3, vehicle_name=DRONE).join()
    time.sleep(1)

    positions = []
    rewards = []

    for step in range(n_steps):
        state = get_drone_state(client, DRONE)
        action = policy_fn(state, target)
        # 执行动作
        client.moveByRollPitchYawrateThrottleAsync(
            float(action[1]) * 0.3,   # roll
            float(action[0]) * 0.3,   # pitch
            float(action[2]) * 0.5,   # yaw_rate
            float(action[3]) * 0.5 + 0.5,  # throttle
            duration=0.1, vehicle_name=DRONE
        ).join()

        state = get_drone_state(client, DRONE)
        reward, dist = compute_hover_reward(state, target)
        positions.append(state['pos'].copy())
        rewards.append(reward)

        if dist > 15:  # 飞太远了
            print(f"  [{label}] 飞出边界，step={step}")
            break

    return np.array(positions), np.array(rewards)

# 策略1：随机动作
def random_policy(state, target):
    return np.random.randn(4) * 0.3

# 策略2：简单PD控制（基于位置误差的比例-微分控制）
def pd_policy(state, target):
    pos_err = target - state['pos']
    vel = state['vel']
    kp, kd = 0.5, 0.3
    control = kp * pos_err - kd * vel
    pitch = np.clip(control[0], -1, 1)
    roll = np.clip(control[1], -1, 1)
    throttle = np.clip(control[2] * 0.5, -1, 1)
    return np.array([pitch, roll, 0.0, throttle])

print("运行 baseline 对比实验...")
print("\n--- 随机动作 ---")
pos_random, rew_random = run_episode(client, random_policy, TARGET, 150, "随机")
print(f"  平均奖励: {rew_random.mean():.3f}, 最终距离: {np.linalg.norm(pos_random[-1] - TARGET):.2f}m")

print("\n--- PD控制 ---")
pos_pd, rew_pd = run_episode(client, pd_policy, TARGET, 150, "PD")
print(f"  平均奖励: {rew_pd.mean():.3f}, 最终距离: {np.linalg.norm(pos_pd[-1] - TARGET):.2f}m")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# XY轨迹
axes[0].plot(pos_random[:, 0], pos_random[:, 1], 'r-', alpha=0.7, label='随机动作')
axes[0].plot(pos_pd[:, 0], pos_pd[:, 1], 'b-', alpha=0.7, label='PD控制')
axes[0].plot(TARGET[0], TARGET[1], 'k*', markersize=15, label='目标')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title('XY平面轨迹'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 高度
axes[1].plot(pos_random[:, 2], 'r-', alpha=0.7, label='随机动作')
axes[1].plot(pos_pd[:, 2], 'b-', alpha=0.7, label='PD控制')
axes[1].axhline(y=TARGET[2], color='k', linestyle='--', label='目标高度')
axes[1].set_xlabel('时间步'); axes[1].set_ylabel('Z (m)')
axes[1].set_title('高度变化'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# 奖励
axes[2].plot(rew_random, 'r-', alpha=0.7, label='随机动作')
axes[2].plot(rew_pd, 'b-', alpha=0.7, label='PD控制')
axes[2].set_xlabel('时间步'); axes[2].set_ylabel('奖励')
axes[2].set_title('每步奖励'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hover_baseline_comparison.png', dpi=150)
plt.show()
print("对比图已保存: hover_baseline_comparison.png")

## 8.3.3 世界模型策略（预训练权重推理）

如果有预训练好的 DreamerV3 权重，加载后直接推理。

如果没有预训练权重（当前状态），我们用一个基于简化世界模型的 MPC（模型预测控制）策略来演示世界模型的核心思想：**在想象中试多种动作，选最好的那个**。

In [ ]:
import torch
from world_model_tools import SimpleWorldModel
import os

# 检查是否有预训练权重
CHECKPOINT = 'models/hover_checkpoint/model.pt'
has_pretrained = os.path.exists(CHECKPOINT)

if has_pretrained:
    print(f"找到预训练权重: {CHECKPOINT}")
    # TODO: 加载 DreamerV3 权重并推理
    # model = DreamerV3.load(CHECKPOINT)
else:
    print("未找到预训练权重，使用简化版 MPC 策略演示")
    print("MPC 策略：在每一步，用世界模型'想象'多种动作的后果，选择奖励最高的动作")

# 简化版 MPC 策略（基于世界模型的在线规划）
def mpc_policy(state_dict, target, n_candidates=50, horizon=5):
    """模型预测控制：采样多个动作序列，用世界模型评估，选最优。"""
    pos = state_dict['pos']
    vel = state_dict['vel']
    best_action = None
    best_reward = -float('inf')

    for _ in range(n_candidates):
        # 随机采样一个动作
        action = np.random.randn(4) * 0.5
        # 简单的物理模型"想象"：预测执行该动作后的位置
        pred_pos = pos + vel * 0.1 + np.array([action[0], action[1], action[3]]) * 0.05
        pred_dist = np.linalg.norm(pred_pos - target)
        pred_reward = max(0, 1.0 - pred_dist / 5.0)

        if pred_reward > best_reward:
            best_reward = pred_reward
            best_action = action

    return best_action

print("\n--- MPC策略（世界模型规划） ---")
pos_mpc, rew_mpc = run_episode(client, mpc_policy, TARGET, 150, "MPC")
print(f"  平均奖励: {rew_mpc.mean():.3f}, 最终距离: {np.linalg.norm(pos_mpc[-1] - TARGET):.2f}m")

In [ ]:
# 三种策略对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for data, label, color in [
    (pos_random, '随机动作', 'r'),
    (pos_pd, 'PD控制', 'b'),
    (pos_mpc, 'MPC(世界模型)', 'g'),
]:
    axes[0].plot(data[:, 0], data[:, 1], f'{color}-', alpha=0.7, label=label)
axes[0].plot(TARGET[0], TARGET[1], 'k*', markersize=15, label='目标')
axes[0].set_title('XY平面轨迹'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for data, label, color in [
    (rew_random, '随机动作', 'r'),
    (rew_pd, 'PD控制', 'b'),
    (rew_mpc, 'MPC(世界模型)', 'g'),
]:
    axes[1].plot(np.cumsum(data), f'{color}-', alpha=0.7, label=label)
axes[1].set_title('累计奖励'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# 距离目标的距离
for data, label, color in [
    (pos_random, '随机动作', 'r'),
    (pos_pd, 'PD控制', 'b'),
    (pos_mpc, 'MPC(世界模型)', 'g'),
]:
    dists = np.linalg.norm(data - TARGET, axis=1)
    axes[2].plot(dists, f'{color}-', alpha=0.7, label=label)
axes[2].set_title('距目标距离'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hover_3way_comparison.png', dpi=150)
plt.show()

In [ ]:
# 拍摄最终悬停状态的截图
responses = client.simGetImages([
    airsim.ImageRequest("0", airsim.ImageType.Scene, False, False)
], vehicle_name=DRONE)

if responses and responses[0].width > 0:
    img = np.frombuffer(responses[0].image_data_uint8, dtype=np.uint8)
    img = img.reshape(responses[0].height, responses[0].width, 3)
    Image.fromarray(img[:,:,::-1]).save('hover_final_view.png')
    print("悬停状态截图已保存: hover_final_view.png")

state = get_drone_state(client, DRONE)
print(f"最终位置: {state['pos']}")
print(f"距目标: {np.linalg.norm(state['pos'] - TARGET):.2f}m")

## 8.3.4 小结

| 策略 | 原理 | 悬停效果 |
|------|------|----------|
| 随机动作 | 无任何智能 | 很差，快速飘走 |
| PD控制 | 经典控制理论 | 较好，但需要手动调参 |
| MPC(世界模型) | 在想象中规划 | 好，自动学习控制策略 |
| DreamerV3(预训练) | 深度世界模型 | 最好（需要预训练权重） |

MPC 策略展示了世界模型的核心价值：**不需要手动设计控制规则，而是通过"想象"不同动作的后果来自动选择最优行动**。

下一节，我们将在更复杂的避障任务中进一步验证这一思想。